<a href="https://colab.research.google.com/github/sandipankar-data-engineer/data-engineering-scenarios-pandas/blob/sample-pandas-rm11062026/ConvertJSON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RoughWork

```
df_list = []
for index,row in df_flat2.iterrows():
  df_norm = pd.json_normalize(row['objects'])
  for index2,row2 in df_norm.iterrows():
    row2['prodno'] = row['partno']
    df_list.append(row2.to_dict())

df_list_value = pd.DataFrame(df_list)

df_flat3 = df_list_value.explode('values')[['prodno','type','values']]

df_flat3['n_value'] = [i['value'] for i in df_flat3['values']]

df_flat3 = df_flat3.drop('values',axis=1)

df_flat3.head(30)
```







```
df = []
object_list(record)
dataf = pd.DataFrame(df, columns=['Column', 'Value'])
dataf = dataf.drop_duplicates()
dataf = dataf.reset_index(drop=True)
dataf.loc[dataf['Column'] == 'Part']['Value'].to_dict()[0]
dataf['Part'] = dataf.loc[dataf['Column'] == 'Part']['Value'].to_dict()[0]
```



# Importing Libraries and JSON Data

In [ ]:
import pandas as pd
import json
import math

In [ ]:
with open('data.json') as json_file:
    json_data = json.load(json_file)

FileNotFoundError: [Errno 2] No such file or directory: 'data.json'

# Recursive Logic to extract data from JSON format

In [ ]:
record = json_data['contents'][0]['objects'][1]

In [ ]:
def object_list(x, name=''):
  global df
  if type(x) is dict:
    for a in x:
      try:
        tdf = pd.json_normalize(x,"values" , meta=["type"])
        df.append([str(tdf['type'].values[0]+name),tdf['value'].values[0]])
      except:
        pass
      if type(x[a]) is list or type(x[a]) is dict:
        if(a != 'values'):
          name = name+":"+a
        object_list(x[a], name)

  elif type(x) is list:
    for a in x:
      object_list(a, name)

In [ ]:
df = []
df_part_lists = pd.DataFrame()
for i in range(0,len(json_data['contents'][0]['objects'])):
  record = json_data['contents'][0]['objects'][i]
  object_list(record)
  dataf = pd.DataFrame(df, columns=['Column', 'Value'])
  dataf = dataf.drop_duplicates()
  dataf = dataf.reset_index(drop=True)
  dataf['Part'] = dataf.loc[dataf['Column'] == 'part']['Value'].to_dict()[0]
  df_part_lists = pd.concat([dataf,df_part_lists])
  df = []

In [ ]:
df_part_lists = df_part_lists[['Part', 'Column', 'Value']]

In [ ]:
df_part_lists = df_part_lists.rename(columns={'Part': 'partno'})
df_part_lists = df_part_lists.rename(columns={'Column': 'type'})
df_part_lists = df_part_lists.rename(columns={'Value': 'n_value'})

In [ ]:
df_part_lists = df_part_lists.drop_duplicates()

In [ ]:
df_part_lists = df_part_lists.reset_index(drop=True)

In [ ]:
dataf_pivot = df_part_lists.pivot(index='partno', columns='type')

In [ ]:
dataf_pivot.to_csv('dataf_pivot.csv')

# JSON Flattening Logic to extract data from JSON

In [ ]:
df = pd.json_normalize(json_data)

In [ ]:
df_flat_contents = pd.json_normalize(df.explode('contents')['contents'])

In [ ]:
df_flat_objects = pd.json_normalize(df_flat_contents.explode('objects')['objects'])

In [ ]:
df_flat_values = df_flat_objects.explode('values')

In [ ]:
df_flat_values["partno"] = [i['value'] for i in df_flat_values['values']]

In [ ]:
df_flat_part_objects = df_flat_values[["partno","objects"]]

In [ ]:
df_flat_nested_objects = df_flat_part_objects.explode('objects')

In [ ]:
df_flat_nested_objects = df_flat_nested_objects.reset_index().drop('index', axis=1)

In [ ]:
df_list_dict = []
for index,row in df_flat_nested_objects.iterrows():
  row['objects']['partno']=row['partno']
  df_list_dict.append(row['objects'])

In [ ]:
df_list_dict_view = pd.DataFrame(df_list_dict)

In [ ]:
df_flat_attributes = df_list_dict_view.explode('values')[['partno','type','values']]

In [ ]:
df_flat_attributes['n_value'] = [i['value'] for i in df_flat_attributes['values']]

In [ ]:
df_flat_attributes = df_flat_attributes.drop('values',axis=1)

In [ ]:
df_flat_attributes

,partno,type,n_value
0,3028936,design_responsible,WHENDR
1,3028936,designgroup,UBPM
2,3028936,unit_of_measure,st
3,3028936,part_status,P
4,3028936,item_name_short,VCB
...,...,...,...
63,3074175,item_name_long,VCB Sealing Overmould
64,3074175,intro_co,720864
65,3074175,estimated_weight,1.0
66,3074175,defining_document,Exist


In [ ]:
df_flat_pivot = df_flat_attributes.pivot(index='partno', columns='type')

In [ ]:
df_flat_pivot

n_value                                                   \
type    calculated_weight defining_document design_responsible designgroup   
partno                                                                       
3028936               NaN             Exist             WHENDR        UBPM   
3060750           66.7482             Exist             AKU2JU        UBPM   
3067076           5.82129             Exist             WHENDR        UBPM   
3067078               NaN             Exist             WHENDR        UBPM   
3074173               NaN             Exist             WHENDR        UBPM   
3074175               NaN             Exist             WHENDR        UBPM   

                                                                    \
type    disc_co estimated_weight intro_co           item_name_long   
partno                                                               
3028936  749337             38.0   720864    VCB Connector Assy LH   
3060750  747971             67.0   720864  End plate outer LH (B3)   
3067076     NaN              6.0   720864          VCB Cover Front   
3067078     NaN              8.0   720864           VCB Cover Rear   
3074173     NaN              3.0   720864   VCB Self Clinching Nut   
3074175     NaN              1.0   720864    VCB Sealing Overmould   

                                                                            \
type    item_name_short last_co part_status replaced_by_part sourcing_area   
partno                                                                       
3028936             VCB  735879           P            Exist           SEU   
3060750       End plate  735879           P            Exist           SEU   
3067076             VCB  720864           P              NaN           NaN   
3067078             VCB  720864           P              NaN           NaN   
3074173             VCB  720864           P              NaN           NaN   
3074175             VCB  720864           P              NaN           NaN   

                         
type    unit_of_measure  
partno                   
3028936              st  
3060750              st  
3067076              st  
3067078              st  
3074173              st  
3074175              st

In [ ]:
df_flat_attributes_2 = df_list_dict_view.explode('values')[['partno','type','values','objects']]

In [ ]:
df_flat_attributes_2['n_value'] = [i['value'] for i in df_flat_attributes_2['values']]

In [ ]:
df_flat_attributes_2 = df_flat_attributes_2.drop('values',axis=1)

In [ ]:
df_flat_attributes_2 = df_flat_attributes_2.explode('objects')

In [ ]:
df_flat_attributes_2.head()

,partno,type,objects,n_value
0,3028936,design_responsible,NaN,WHENDR
1,3028936,designgroup,NaN,UBPM
2,3028936,unit_of_measure,"{'type': 'item_name_long', 'displaytypename': ...",st
3,3028936,part_status,NaN,P
4,3028936,item_name_short,NaN,VCB


In [ ]:
dict_all_atrib = []
for index, row in df_flat_attributes_2.iterrows():
  new_dict = {}
  for i in row.keys():
    if i != 'objects':
      new_dict[i] = row.loc[i]
    elif type(row.loc['objects']) is dict and not(pd.isnull(row.loc[i])):
      new_dict.update(row.loc['objects'])
      print(row.loc['objects'])
  dict_all_atrib.append(new_dict)

{'type': 'item_name_long', 'displaytypename': 'Item Name Long', 'displaytypenamelang': 'en', 'values': [{'lang': 'en', 'value': 'Pieces'}]}
{'type': 'eco', 'alias': 'eco', 'displaytypename': 'ECO', 'displaytypenamelang': 'en', 'values': [{'value': 720864}], 'objects': [{'type': 'time_carrier', 'displaytypename': 'Time Carrier', 'displaytypenamelang': 'en', 'values': [{'value': 129838}], 'objects': [{'type': 'socop', 'displaytypename': 'SOCOP', 'displaytypenamelang': 'en', 'values': [{'value': '2023.11.1'}]}]}]}
{'type': 'item_name_long', 'displaytypename': 'Item Name Long', 'displaytypenamelang': 'en', 'values': [{'lang': 'en', 'value': 'Pieces'}]}
{'type': 'eco', 'alias': 'eco', 'displaytypename': 'ECO', 'displaytypenamelang': 'en', 'values': [{'value': 720864}], 'objects': [{'type': 'time_carrier', 'displaytypename': 'Time Carrier', 'displaytypenamelang': 'en', 'values': [{'value': 129838}], 'objects': [{'type': 'socop', 'displaytypename': 'SOCOP', 'displaytypenamelang': 'en', 'value

In [ ]:
pd.DataFrame(dict_all_atrib)

,partno,type,n_value,displaytypename,displaytypenamelang,values,alias,objects
0,3028936,design_responsible,WHENDR,NaN,NaN,NaN,NaN,NaN
1,3028936,designgroup,UBPM,NaN,NaN,NaN,NaN,NaN
2,3028936,item_name_long,st,Item Name Long,en,"[{'lang': 'en', 'value': 'Pieces'}]",NaN,NaN
3,3028936,part_status,P,NaN,NaN,NaN,NaN,NaN
4,3028936,item_name_short,VCB,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
63,3074175,item_name_long,VCB Sealing Overmould,NaN,NaN,NaN,NaN,NaN
64,3074175,eco,720864,ECO,en,[{'value': 720864}],eco,"[{'type': 'time_carrier', 'displaytypename': '..."
65,3074175,estimated_weight,1.0,NaN,NaN,NaN,NaN,NaN
66,3074175,defining_document,Exist,NaN,NaN,NaN,NaN,NaN
